In [1]:
import sys
import time
import initial_vars as var
import copy

In [2]:
dblMoves, movements, starting_positions, piece_types = var.dblMoves, var.movements, var.startingPositions, var.piece_types

In [50]:
#Tuple/index/Bit operations
def bitIndex(tuple):
    index = tuple[0] * 8 + tuple[1]
    return index

def indexToTuple(index):
    row = index // 8
    col = index % 8
    return row, col

def changeBit(b, i, bool):
    if bool == True:
        b = b[:i] + '1' + b[i + 1:]
    elif bool == False:
        b = b[:i] + '0' + b[i + 1:]
    return b

def indexToPos(bin):
    tups = []
    for i in range(64):
        if bin[i] == '1':
            tups.append(indexToTuple(i))
    return tups


# create & comprehend board
def bit_initialise(positions):
    board = {}
    for item, posns in positions.items():
        binary = '0'*64
        for position in posns:
            binary = changeBit(binary, bitIndex(position), True)
        board[item] = binary
    return board

# comprehend coloured boards
def colourDict(boardState):
    dict_white = {}
    dict_black = {}
    for piece in boardState:
        if piece in ['K', 'Q', 'R', 'B', 'N', 'P']:
            dict_white[piece] = boardState[piece]
        elif piece in ['k', 'q', 'r', 'b', 'n', 'p']:
            dict_black[piece] = boardState[piece]
    return dict_white, dict_black

def setActive(piece, boardState):
    w, b = colourDict(boardState)
    if piece in w:
        active, inactive = w, b
    else:
        active, inactive = b, w
    return active, inactive


def pieceInIndex(index, dict):
    for piece in dict:
        if (dict[piece][index]) == '1':
            return piece
    return None

def setActiveByColor(color, boardState):
    w, b = colourDict(boardState)
    if color == 'w':
        return w
    elif color == 'b':
        return b

def get_newPos(position, move):
    return (position[0] + move[0], position[1] + move[1])

def genEdges():
    edges = []
    for i in range(8):
        for j in range(8):
            if (i == 0) or (i == 7):
                edges.append((i, j))
            elif (j==0) or (j==7):
                edges.append((i,j))
    return edges


def onBoard(pos):
    if (pos[0] in range(8)) and (pos[1] in range(8)):
        return True
    return False


def isCaptured(piece, move, boardState):
    inactive = setActive(piece, boardState)[1]
    for opp in inactive:
        if boardState[opp][move[1]] == '1':
            return opp
    return False

def isThreatened(piece, boardState, dblMoveIndex):
    active, inactive = setActive(piece, boardState)
    threats = []
    for opp in inactive:
        captures = moves_postBlock(opp, boardState, False, dblMoveIndex)[1]
        caps = len(captures)
        if caps > 0:
            for i in range(caps):
                if active[piece][captures[i][1]] == '1':
                    threats.append(captures[i])
    return threats

def isCheck(active, boardState, dblMoveIndex):
    if 'k' in active:
        if len(isThreatened('k', boardState, dblMoveIndex)) > 0:
            return 'black in check', isThreatened('k', boardState, dblMoveIndex)
        else:
            return False
    elif 'K' in active:
        if len(isThreatened('K', boardState, dblMoveIndex)) > 0:
            return 'white in check', isThreatened('K', boardState, dblMoveIndex)
        else:
            return False

def moves_postBlock(piece, boardState, audit, dblMoveIndex):
    """
    returns two lists of tuples, where the tuples are (old_position, new_position). 
    1st list: non-capture moves
    2nd list: capture moves

    params::
    piece: string, e.g. 'Q' or 'q', as in startingPositions & movements (Uppercase = White, Lowercase = Black)
    boardState: the boardState dict of {piece: binary number} (binary represents position(s))
    movements:  dictionary containing piece movement rules
    audit: boolean; set to True to assess step-by-step evaluations
    """
    if audit: print(piece)
    edges = genEdges()
    all_positions = indexToPos(boardState[piece])
    legal, captures = [], []
    active, inactive = setActive(piece, boardState)
    for position in all_positions:        
        if audit: print(position)
        if (piece == 'P') or (piece == 'p'):
            dbl = False
            startingPawns = pawn_starting_double(piece, boardState)
            fwd = []
            forwardSquare = movements[piece][0][0]
            captureSquares = movements[piece][1]
            if dblMoveIndex != None:
                if position not in edges:
                    if (bitIndex(position) + 1 == dblMoveIndex) or (bitIndex(position) - 1 == dblMoveIndex):
                        if piece == 'P':
                            captures.append((position, indexToTuple(dblMoveIndex + 8)))
                        if piece == 'p':
                            captures.append((position, indexToTuple(dblMoveIndex - 8)))
                else:
                    if onBoard(indexToTuple(bitIndex(position) + 1)):
                        if bitIndex(position) + 1 == dblMoveIndex:
                            if piece == 'P':
                                captures.append((position, indexToTuple(dblMoveIndex + 8)))
                            if piece == 'p':
                                captures.append((position, indexToTuple(dblMoveIndex - 8)))
                    elif onBoard(indexToTuple(bitIndex(position) - 1)):
                        if bitIndex(position) - 1 == dblMoveIndex:
                            if piece == 'P':
                                captures.append((position, indexToTuple(dblMoveIndex + 8)))
                            if piece == 'p':
                                captures.append((position, indexToTuple(dblMoveIndex - 8)))
            fwd.append(get_newPos(position, forwardSquare))
            if bitIndex(position) in startingPawns:
                dbl = True
                if piece == 'p':
                    fwd.append(get_newPos(position, (-2, 0)))
                elif piece == 'P':
                    fwd.append(get_newPos(position, (2, 0)))
            cap1, cap2 = get_newPos(position, captureSquares[0]), get_newPos(position, captureSquares[1])
            if dbl:
                if pieceInIndex(bitIndex(fwd[0]), boardState) == None:
                    if pieceInIndex(bitIndex(fwd[1]), boardState) == None:
                        legal.append((position, fwd[1]))
                    legal.append((position, fwd[0]))
            elif not dbl:
                if pieceInIndex(bitIndex(fwd[0]), boardState) == None:
                    legal.append((position, fwd[0]))
            if pieceInIndex(bitIndex(cap1), boardState) in inactive:
                captures.append((position, cap1))
            if pieceInIndex(bitIndex(cap2), boardState) in inactive:
                captures.append((position, cap2))
        else:
            for move_sect in movements[piece]:
                block, capture, block, edge, offBoard = False, False, False, False, False
                for i, move in enumerate(move_sect):
                    newPos = get_newPos(position, move)
                    if audit: print(newPos)
                    if onBoard(newPos):
                        squareCheck = pieceInIndex(bitIndex(newPos), boardState)
                        if squareCheck != None:
                            if squareCheck in inactive:
                                if audit: print('capture') 
                                [legal.append((position, get_newPos(position, mv))) for mv in move_sect[:i]]
                                captures.append((position, newPos))
                                capture = True
                                break
                            elif squareCheck in active:
                                if audit: print('block') 
                                [legal.append((position, get_newPos(position, mv))) for mv in move_sect[:i]]
                                block = True
                                break
                        if newPos in edges:
                            if audit: print('edge') 
                            [legal.append((position, get_newPos(position, mv))) for mv in move_sect[:i+1]]
                            edge = True
                            break
                    else:
                        if audit: print('offboard') 
                        offBoard = True
                        break
                if (not block) and (not capture) and (not edge) and (not offBoard):
                    if audit: print('not') 
                    [legal.append((position, get_newPos(position, mv))) for mv in move_sect]
    if audit: print('moves generated')
    legal_index, capture_index = [], []
    for pair in legal:
        if audit: print(pair)
        legal_index.append((bitIndex(pair[0]), bitIndex(pair[1])))
    for pair in captures:
        if audit: print(pair)
        capture_index.append((bitIndex(pair[0]), bitIndex(pair[1])))
    if audit: print(legal_index, capture_index)
    return legal_index, capture_index

def moves_postCheck(piece, boardState, audit, dblMoveIndex):
    legal1, capture1 = moves_postBlock(piece, boardState, audit, dblMoveIndex)
    legal2, capture2 = [], []
    active = setActive(piece, boardState)[0]  
    for move in legal1:
        newboard = boardAfterMove(piece, move, boardState, audit)
        check = isCheck(active, newboard, dblMoveIndex)
        if not check:
            legal2.append(move)
    for move in capture1:
        newboard = boardAfterMove(piece, move, boardState, audit)
        check = isCheck(active, newboard, dblMoveIndex)
        if not check:
            capture2.append(move)
        # paste
    if audit: print("postCheck: ",legal2, capture2)
    return legal2, capture2
        

def boardAfterMove(piece, move, boardState, audit):
    # move is one of the tuples in lists provided by 
    newBoard = copy.deepcopy(boardState)
    victim = isCaptured(piece, move, newBoard)
    if audit: print(boardState[piece], move[0])
    newBoard[piece] = changeBit(newBoard[piece], move[0], False)
    if audit: print(boardState[piece], move[0])
    newBoard[piece] = changeBit(newBoard[piece], move[1], True)
    if audit: print(boardState[piece], move[1])
    if victim != False:
        if audit: print(newBoard[victim], move[1])
        newBoard[victim] = changeBit(newBoard[victim], move[1], False)
    return newBoard

def pawn_starting_double(piece, boardState):
    starting_board = bit_initialise(starting_positions)[piece]
    current = boardState[piece]
    starting = []
    for i in range(64):
        if (current[i] == '1') & (starting_board[i] == '1'):
            starting.append(i)
    return starting

def pawnDoubleMove_index(color, board, move, dblMoves):
    if color == 'w':
        if ('P', move) in dblMoves:
            return move[1]
    elif color == 'b':
        if ('p', move) in dblMoves:
            return move[1]
    return None        



def all_moves(activeColor, boardState, audit, dblMoveIndex):
    nonCaptures, captures = [], []
    for piece in setActiveByColor(activeColor, boardState):
        if audit: print(piece)
        moves = moves_postCheck(piece, boardState, False, dblMoveIndex)
        for a in moves[0]:
            if audit: print(a)
            nonCaptures.append((piece, a))
        for b in moves[1]:
            captures.append((piece, b))
    return nonCaptures, captures
 
def affectMove(color, move, boardState, audit, dblMoves, dblIndex):
    active = setActiveByColor(color, boardState)
    dblMoveIndex = pawnDoubleMove_index(color, boardState, move, dblMoves)
    moves, captures = all_moves(color, boardState, audit, dblIndex)[0], all_moves(color, boardState, audit, dblIndex)[1]
    for mv in moves:
        if mv[1] == move: 
            board_after_move = boardAfterMove(mv[0], mv[1], boardState, audit)
    for mv in captures:
        if mv[1] == move: 
            board_after_move = boardAfterMove(mv[0], mv[1], boardState, audit)
    return board_after_move, dblMoveIndex

def displayBoard(boardState, whiteOnTop):
    empty = "-"
    squares = []
    for piece in boardState:
        for pos in range(64):
            if boardState[piece][pos] == '1':
                squares.append((piece, pos))
    posns = []
    for i in range(64):
        isin = False
        for p in squares:
            if p[1] == i:
                posns.append(p[0])
                isin = True
                break
        if not isin:
            posns.append(empty)
    if whiteOnTop:
        for row in range(8):
            r = f'{posns[8*row + 0]}' + ' | ' + f'{posns[8*row + 1]}' + ' | ' + f'{posns[8*row + 2]}' + ' | ' + f'{posns[8*row + 3]}' + ' | ' + f'{posns[8*row + 4]}' + ' | ' + f'{posns[8*row + 5]}' + ' | ' + f'{posns[8*row + 6]}' + ' | ' + f'{posns[8*row + 7]}'
            print(r)
    elif not whiteOnTop:
        for row in range(7,  -1, -1):
            r = f'{posns[8*row + 0]}' + ' | ' + f'{posns[8*row + 1]}' + ' | ' + f'{posns[8*row + 2]}' + ' | ' + f'{posns[8*row + 3]}' + ' | ' + f'{posns[8*row + 4]}' + ' | ' + f'{posns[8*row + 5]}' + ' | ' + f'{posns[8*row + 6]}' + ' | ' + f'{posns[8*row + 7]}'
            print(r)


# notation comprehension
def filerank_index(filerank):
    return bitIndex((int(filerank[1]) - 1,ord(filerank[0]) - ord('a') ))

def index_filerank(index):
    tup = indexToTuple(index)
    return (chr(tup[1] + ord('a')) + str(tup[0] + 1))


In [ ]:
audit = False
dblMoveIndex = None
whiteOnTop = False
board = bit_initialise(starting_positions)

board, dblMoveIndex = affectMove('w', (8, 24), board, audit, dblMoves, dblMoveIndex)
board, dblMoveIndex = affectMove('b', (51, 35), board, audit, dblMoves, dblMoveIndex)
board, dblMoveIndex = affectMove('w', (24, 32), board, audit, dblMoves, dblMoveIndex)
board, dblMoveIndex = affectMove('b', (49, 33), board, audit, dblMoves, dblMoveIndex)
board, dblMoveIndex = affectMove('w', (32, 41), board, audit, dblMoves, dblMoveIndex)
displayBoard(board, whiteOnTop)

In [2]:
for i in range(7):
    print((chr(ord('a') + i)).upper())

A
B
C
D
E
F
G


: 